# EFA Fitting and Checks

Runs EFA and post-EFA diagnostics. Edit the **Paths** cell before running.

**Post-EFA:** communalities, variance explained, factor loadings, and cross-loadings

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from langdetect import detect, LangDetectException
from collections import defaultdict
from sklearn.preprocessing import PowerTransformer
from factor_analyzer.factor_analyzer import calculate_kmo, calculate_bartlett_sphericity, FactorAnalyzer
import joblib

from ep_pipeline.config import MetricsConfig
from ep_pipeline.io import load_csv, write_table
from ep_pipeline.efa.efa import prepare_features, scale_metrics, parallel_analysis, plot_scree, fit_efa
from ep_pipeline.efa.metric_taxonomy import CATEGORY_OF, METRIC_DOMAIN_OF

## Paths

In [ ]:
PROJECT_ROOT = Path.cwd().parent  # nbs/ is one level under chr_stability_paper/
TEXT_FP_DETECT = PROJECT_ROOT / 'data' / 'human_texts_2023_2025.csv'
CHR27_FP       = PROJECT_ROOT / 'data' / 'data_subset_chr27.csv'
IN_FP       = PROJECT_ROOT / 'data' / 'AO3metrics_full.csv'
VIS_DIR     = PROJECT_ROOT / 'outputs' / 'visualizations'
RESULTS_DIR = PROJECT_ROOT / 'outputs' / 'results' / 'efa_assumptions'
VIS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

cfg = MetricsConfig()

In [ ]:
# Run the imports cell + paths cell, then this cell instead of check_pre_efa_assumption.ipynb
STATE_PATH = PROJECT_ROOT / 'data' / 'efa_state.joblib'

if not STATE_PATH.exists():
    import subprocess
    print(f"{STATE_PATH.name} not found - running check_pre_efa_assumptions.ipynb to generate it.")
    print("(That notebook falls back to the anonymized final_full_dataset.csv automatically")
    print("if the scoring pipeline hasn't been run either, so this works either way.)\n")
    subprocess.run(
        ['python3', '-m', 'nbconvert', '--to', 'notebook', '--execute', '--inplace',
         'check_pre_efa_assumptions.ipynb', '--ExecutePreprocessor.timeout=1800'],
        cwd=PROJECT_ROOT / 'nbs', check=True,
    )
    print()

_s = joblib.load(STATE_PATH)

scaled_metrics     = _s['scaled_metrics']
scaled_clean       = _s['scaled_clean']
corr_clean         = _s['corr_clean']
feature_cols       = _s['feature_cols']
keep_cols          = _s['keep_cols']
final_category_map = _s['final_category_map']
category_clean     = _s['category_clean']
n_obs              = _s['n_obs']
n_vars             = _s['n_vars']
meta_full          = _s['meta_full']
used_fallback       = _s.get('used_fallback', False)

print(f"Loaded from {STATE_PATH}")
print(f"  scaled_metrics : {scaled_metrics.shape}")
print(f"  scaled_clean   : {scaled_clean.shape}  ({len(keep_cols)} vars)")
print(f"  meta_full      : {meta_full.shape}  (same row order as scaled_metrics/scaled_clean)")
if used_fallback:
    print("\n  NOTE: this state was built from the anonymized final_full_dataset.csv fallback,")
    print("  not the raw scored pipeline output - refitting EFA here reproduces results on an")
    print("  already-reduced feature set. final_full_dataset.csv will not be overwritten below.")

In [ ]:
print(f"Metrics remaining after all cleaning steps: {len(keep_cols)}\n")

hierarchy = defaultdict(lambda: defaultdict(list))
for c in keep_cols:
    hierarchy[METRIC_DOMAIN_OF[c]][category_clean[c]].append(c)

for dom in sorted(hierarchy):
    dom_vars = [c for cat_vars in hierarchy[dom].values() for c in cat_vars]
    print(f"{dom} ({len(dom_vars)}):")
    for cat in sorted(hierarchy[dom]):
        vars_ = sorted(hierarchy[dom][cat])
        print(f"  {cat} ({len(vars_)}):")
        for v in vars_:
            print(f"    {v}")
    print()

## 1. Factor number — parallel analysis & scree

Retains factors whose eigenvalues exceed the 95th percentile from random data of the same shape.  
The Kaiser criterion (eigenvalue > 1) is shown for reference.

In [ ]:
eigenvalues, pa_threshold, n_factors = parallel_analysis(
    scaled_clean, seed=cfg.seed, n_perm=cfg.n_permutations, percentile=99
)
n_kaiser = int((eigenvalues > 1).sum())
print(f'Parallel analysis suggests : {n_factors} factor(s)')
print(f'Kaiser criterion (eig > 1) : {n_kaiser} factor(s)')
plot_scree(eigenvalues, pa_threshold, n_factors, viz_path=VIS_DIR / 'efa_scree.png')
plt.show()


In [ ]:
prev = 0
for k in range(1, 19):
    fa_k = FactorAnalyzer(n_factors=k, rotation='oblimin', method='minres')
    fa_k.fit(scaled_clean)
    var = fa_k.get_factor_variance()
    cumvar = var[2][-1] * 100
    print(f"k={k:2d}  cumulative variance: {cumvar:.1f}%  gain: +{cumvar - prev:.1f}%")
    prev = cumvar

10 factors chosen because parallel analysis supported up to 16 factors, but solution quality (weak/cross-loadings, communalities) plateaued by k=10-11, so we retained 10 factors for interpretability without sacrificing fit.

## 2. Fit EFA

Uses `n_factors` from parallel analysis above. Uncomment the override line if the scree plot or theory suggests a different number.

In [ ]:
n_factors = 10

fa, loadings_df, var_df = fit_efa(
    scaled_clean, keep_cols, final_category_map, n_factors
)
print(var_df.to_string(index=False))
total_var = var_df['cumulative_pct'].iloc[-1]
var_pass  = 'PASS' if total_var >= 50 else 'NOTE — below 50%'
print(f'\nTotal variance explained: {total_var:.1f}%  ({var_pass})')

## 8. Communalities

Variables with `h2 < 0.10` are poorly explained by the retained factors so removed and refit without

In [ ]:
communalities = fa.get_communalities()
comm_series   = pd.Series(communalities, index=keep_cols).sort_values()
low_comm      = comm_series[comm_series < 0.1]
print(f'Low communalities (h2 < 0.10): {len(low_comm)} variable(s)')
if not low_comm.empty:
    print(low_comm.to_string())

fig, ax = plt.subplots(figsize=(5, max(6, n_vars // 4)))
colors  = ['tomato' if v < 0.1 else 'steelblue' for v in comm_series.values]
ax.barh(comm_series.index, comm_series.values, color=colors)
ax.axvline(0.1, color='tomato', linestyle='--', linewidth=1, label='h2 = 0.10')
ax.set_xlabel('Communality (h2)')
ax.set_title('Communalities — post-EFA')
ax.legend()
plt.tight_layout()
plt.savefig(VIS_DIR / 'efa_communalities.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Factor loadings & cross-loadings

- **Weak loaders** (`max |loading| < 0.20`): item does not load sufficiently on any factor — consider dropping  
- **Cross-loaders** (2nd highest loading > 75% of highest): item is ambiguous across factors — consider dropping

In [ ]:
factor_col_names = [f'F{i+1}' for i in range(n_factors)]
loadings_vals    = loadings_df[factor_col_names]

annot = n_vars <= 40
fig, ax = plt.subplots(figsize=(max(6, n_factors + 2), max(8, n_vars // 3)))
sns.heatmap(
    loadings_vals, cmap='coolwarm', center=0, vmin=-1, vmax=1,
    annot=annot, fmt='.2f', linewidths=0.3, ax=ax,
    cbar_kws={'shrink': 0.6}
)
ax.set_title('Factor loadings (pattern matrix)')
plt.yticks(fontsize=7)
plt.tight_layout()
plt.savefig(VIS_DIR / 'efa_loadings_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
abs_loadings = loadings_vals.abs()
max_loading  = abs_loadings.max(axis=1)

weak_loaders = max_loading[max_loading < 0.2]
print(f'Weak loaders (max |loading| < 0.20): {len(weak_loaders)} variable(s)')
if not weak_loaders.empty:
    print(weak_loaders.to_string())

cross_loaders = []
for var in abs_loadings.index:
    row = abs_loadings.loc[var].sort_values(ascending=False)
    if n_factors >= 2 and row.iloc[1] > 0.75 * row.iloc[0]:
        cross_loaders.append({
            'variable':          var,
            'primary_factor':    row.index[0],
            'primary_loading':   row.iloc[0],
            'secondary_factor':  row.index[1],
            'secondary_loading': row.iloc[1],
        })

print(f'\nCross-loaders (2nd loading > 75% of 1st): {len(cross_loaders)} variable(s)')
if cross_loaders:
    print(pd.DataFrame(cross_loaders).to_string(index=False))

## Drop poor variables and refit

Drop variables that appear in **both** weak loaders and low communality. Refit on `scaled_clean` / `keep_cols` (67 vars)

In [ ]:
drop_iter1 = {
    'dash_rate', 'semicolon_rate', 'prevalence_mean'}

actually_dropped = drop_iter1 & set(keep_cols)
keep_cols2       = [c for c in keep_cols if c not in drop_iter1]
keep_idx2        = [keep_cols.index(c) for c in keep_cols2]
scaled_clean2    = scaled_clean[:, keep_idx2]

print(f"Dropped  : {sorted(actually_dropped)}")
print(f"Remaining: {len(keep_cols2)} variables  (was {len(keep_cols)})")

In [ ]:
N_FACTORS2 = 10

fa2 = FactorAnalyzer(n_factors=N_FACTORS2, rotation='oblimin', method='minres')
fa2.fit(scaled_clean2)

loadings2    = pd.DataFrame(
    fa2.loadings_,
    index=keep_cols2,
    columns=[f'F{i+1}' for i in range(N_FACTORS2)]
)
var2         = fa2.get_factor_variance()
cumvar2      = var2[2][-1] * 100
print(f"EFA refit: {N_FACTORS2} factors, {len(keep_cols2)} variables")
print(f"Cumulative variance explained: {cumvar2:.1f}%")

In [ ]:
# Communalities
comm2        = pd.Series(fa2.get_communalities(), index=keep_cols2).sort_values()
low_comm2    = comm2[comm2 < 0.1]
print(f"Low communalities (h2 < 0.10): {len(low_comm2)} variable(s)")
if not low_comm2.empty:
    print(low_comm2.to_string())

# Weak loaders
factor_cols2 = [f'F{i+1}' for i in range(N_FACTORS2)]
abs_load2    = loadings2[factor_cols2].abs()
max_load2    = abs_load2.max(axis=1)
weak2        = max_load2[max_load2 < 0.2]
print(f"\nWeak loaders (max |loading| < 0.20): {len(weak2)} variable(s)")
if not weak2.empty:
    print(weak2.to_string())

# Cross-loaders
cross2 = []
for var in abs_load2.index:
    row = abs_load2.loc[var].sort_values(ascending=False)
    if row.iloc[1] > 0.75 * row.iloc[0]:
        cross2.append({
            'variable':          var,
            'primary_factor':    row.index[0],
            'primary_loading':   round(row.iloc[0], 6),
            'secondary_factor':  row.index[1],
            'secondary_loading': round(row.iloc[1], 6),
        })
print(f"\nCross-loaders (2nd loading > 75% of 1st): {len(cross2)} variable(s)")
if cross2:
    print(pd.DataFrame(cross2).to_string(index=False))

## Factor exploration

Salient variables (|loading| > 0.30) for each factor, sorted by magnitude. Use these to name the factors.

In [ ]:
LOADING_THRESHOLD = 0.30

var2_arr = fa2.get_factor_variance()

for i, f in enumerate(factor_cols2):
    salient = loadings2[f][loadings2[f].abs() > LOADING_THRESHOLD].sort_values(key=abs, ascending=False)
    pct     = var2_arr[1][i] * 100
    print(f"\n{'='*72}")
    print(f"  {f}  |  {pct:.1f}% variance  |  {len(salient)} salient variables (|λ| > {LOADING_THRESHOLD})")
    print(f"{'='*72}")
    if len(salient) == 0:
        print("  (no variables above threshold)")
    for var, val in salient.items():
        cat = category_clean.get(var, '?')
        print(f"  {val:+.3f}  {var:<55}  [{cat}]")

## Reporting tables

Two tables for the manuscript:
1. **Factor summary** (main text) — one row per factor: name, % variance, and its top salient loadings.
2. **Full loadings** (supplementary) — all variables × all factors, loadings below `LOADING_THRESHOLD` blanked for readability, rows grouped by primary factor and sorted by loading magnitude within each group.

In [ ]:
# Interpretive names — keep in sync with the factor interpretation markdown below
FACTOR_NAMES = {
    'F1':  'Syntactic Complexity',
    'F2':  'Lexical Sophistication',
    'F3':  'Negative Affect',
    'F4':  'Lexical Richness',
    'F5':  'Repetitiveness',
    'F6':  'Concreteness',
    'F7':  'Conversationality',
    'F8':  'Positive Affect',
    'F9':  'Narrative Drift',
    'F10': 'Structural Variability',
}

TOP_N = 5  # max salient variables to show per factor in the main-text table

summary_rows = []
for i, f in enumerate(factor_cols2):
    salient = loadings2[f][loadings2[f].abs() > LOADING_THRESHOLD].sort_values(key=abs, ascending=False)
    top     = salient.head(TOP_N)
    top_str = '; '.join(f'{var} ({val:+.2f})' for var, val in top.items())
    n_more  = len(salient) - len(top)
    if n_more > 0:
        top_str += f'; +{n_more} more (see Supplementary Table)'

    summary_rows.append({
        'Factor':         f,
        'Name':           FACTOR_NAMES.get(f, ''),
        '% Variance':     round(var2_arr[1][i] * 100, 1),
        'Cumulative %':   round(var2_arr[2][i] * 100, 1),
        'Top loadings':   top_str,
    })

factor_summary_table = pd.DataFrame(summary_rows)
write_table(factor_summary_table, RESULTS_DIR / 'efa_factor_summary_table.csv')
print(f"Saved → {RESULTS_DIR / 'efa_factor_summary_table.csv'}")
factor_summary_table

In [ ]:
# Full pattern matrix — every variable x every factor, for supplementary materials
loadings_all      = loadings2[factor_cols2]
primary_factor    = loadings_all.abs().idxmax(axis=1)
primary_loading   = loadings_all.abs().max(axis=1)
primary_factor_no = primary_factor.str.replace('F', '', regex=False).astype(int)

row_order = (
    pd.DataFrame({'factor_no': primary_factor_no, 'loading': primary_loading})
    .sort_values(['factor_no', 'loading'], ascending=[True, False])
    .index
)

# blank loadings below threshold so the table is scannable, not a wall of numbers
full_loadings_table = loadings_all.loc[row_order].round(2)
full_loadings_table = full_loadings_table.where(full_loadings_table.abs() > LOADING_THRESHOLD)
full_loadings_table.insert(0, 'category', [category_clean.get(v, '?') for v in row_order])
full_loadings_table.insert(0, 'primary_factor', primary_factor.loc[row_order])
full_loadings_table.index.name = 'variable'

write_table(full_loadings_table.reset_index(), RESULTS_DIR / 'efa_full_loadings_table.csv')
print(f"Saved → {RESULTS_DIR / 'efa_full_loadings_table.csv'}")
print(f"{len(full_loadings_table)} variables x {len(factor_cols2)} factors")
print("Note: loadings <= LOADING_THRESHOLD are blank; bold the primary_factor loading per row when formatting for print.")
full_loadings_table

## Factor scores

Compute one score per factor per text using the fitted `fa2`, then save.

In [ ]:
META_COLS = ['id', 'title', 'author', 'fandom', 'published', 'rating',
             'words', 'chapters', 'fandom_label', 'source']

# meta_full was loaded from efa_state.joblib — same row order as scaled_clean/scaled_clean2,
# already covers both source CSVs (chr27 + human_texts_2023_2025) since AO3metrics_full.csv does.
meta = meta_full[META_COLS].reset_index(drop=True)

# Compute factor scores — shape (n_texts, n_factors)
factor_scores = fa2.transform(scaled_clean2)
scores_df     = pd.DataFrame(factor_scores, columns=factor_cols2)

# Merge metadata + scores
result_df = pd.concat([meta, scores_df], axis=1)
print(f"Factor scores : {scores_df.shape}  (texts × factors)")
print(f"Result df     : {result_df.shape}  (texts × meta + factors)")

SCORES_PATH = PROJECT_ROOT / 'data' / 'efa_factor_scores.csv'
result_df.to_csv(SCORES_PATH, index=False)
print(f"\nSaved → {SCORES_PATH}")

result_df.head()

In [ ]:
# Full pipeline output: all metadata + the cleaned/scaled variables + the 10 factor scores.
# Row order matches meta_full/scaled_clean/scaled_clean2/factor_scores throughout.
FINAL_FULL_PATH = PROJECT_ROOT / 'data' / 'final_full_dataset.csv'

final_full_df = pd.concat(
    [meta_full.reset_index(drop=True),
     pd.DataFrame(scaled_clean2, columns=keep_cols2),
     scores_df],
    axis=1
)

if used_fallback and FINAL_FULL_PATH.exists():
    # This run started from final_full_dataset.csv itself (no scoring pipeline output was
    # available), so writing back here would replace the authoritative anonymized results
    # with a refit-on-already-reduced-data approximation. Save it under a different name
    # instead so nothing gets silently overwritten.
    REPRO_PATH = PROJECT_ROOT / 'data' / 'final_full_dataset_reproduced_from_fallback.csv'
    write_table(final_full_df, REPRO_PATH)
    print(f"NOTE: not overwriting {FINAL_FULL_PATH.name} - this run started from it "
          f"(anonymized fallback), so it's a reproduction, not a fresh result.")
    print(f"Saved reproduction attempt: {final_full_df.shape}  "
          f"({meta_full.shape[1]} meta + {len(keep_cols2)} vars + {len(factor_cols2)} factor scores)")
    print(f"Saved → {REPRO_PATH}")
else:
    write_table(final_full_df, FINAL_FULL_PATH)
    print(f"Final full dataset: {final_full_df.shape}  "
          f"({meta_full.shape[1]} meta + {len(keep_cols2)} vars + {len(factor_cols2)} factor scores)")
    print(f"Saved → {FINAL_FULL_PATH}")

### Factor	Name	% var	Interpretation (high score = more of...)

F1	Syntactic Complexity	8.5%	subordination/coordination, parse depth, sentence-length variability, readability (sentence-length component)

F2	Lexical Sophistication	6.6%	syllable length, AoA, harder-vocabulary readability indices, fewer pronouns/CCONJ

F3	Negative Affect	6.6%	anger, fear, sadness, disgust, valence variance, low mean valence

F4	Lexical Richness	5.8%	unique tokens, topic strength, semantic volume, max cluster share, low duplication

F5	Repetitiveness	5.5%	top n-gram fractions, duplication, OOV, low alpha-ratio, fewer modifiers

F6	Concreteness	5.2%	concreteness, nouns, low auxiliaries/particles/adverbs

F7	Conversationality	5.1%	interjections, questions, quotations, exclamations, dependency-distance variability

F8	Positive Affect	3.7%	joy, trust, anticipation, positive valence

F9	Narrative Drift	3.5%	semantic dispersion, drift, transition variability

F10	Structural Variability	3.2%	sentence/dependency/parse-depth std, mixed POS signal*

\* overlaps with F1 (`sentence_length_std`) and F7 (`dependency_distance_std`), which are also salient there so likely picking up leftover variability variance rather than a fully distinct construct.